# Seal Detection!

This notebook walks you through:
1. Transfer learning: Take the weights from a pre-existing model and learn new weights for a new context.
2. Inference and evaluation of a model

Run the following cells each time you restart your kernel or Jupyter server.

In [1]:
# silence warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

In [2]:
%pip install opencv-python -q
%pip install keras_cv -q
%pip install pycocotools -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import keras
import keras_cv
from keras.callbacks import ModelCheckpoint
from keras_cv import visualization

Do you have access to a GPU? The following cell will tell you! If you do, then training tends to go *a lot* faster.

In [4]:
# ensure access to GPU
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

## Read in the Data
You can check out the data by visiting the `shared/seals` directory. The images have been tiled and split into train, validation, and test sets already. In each set, you will find the images and a CSV file containing the *ground truth annotations* of seals. Seals are labeled according to their age/sex: "cow," "pup," and "bull."

We need the data in a particular format for training and inference, so that's what is read in below:

In [5]:
def parse_fn(example_proto):
    feature_description = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'boxes': tf.io.FixedLenFeature([], tf.string),
        'classes': tf.io.FixedLenFeature([], tf.string),
    }
    features = tf.io.parse_single_example(example_proto, feature_description)
    
    # Decode the tensors and set their shapes explicitly
    image = tf.io.parse_tensor(features['image'], out_type=tf.float32)
    image.set_shape([640, 640, 3])
    
    boxes = tf.io.parse_tensor(features['boxes'], out_type=tf.float32)
    boxes.set_shape([32, 4])
    
    classes = tf.io.parse_tensor(features['classes'], out_type=tf.float32)
    classes.set_shape([32])
    
    return image, {"boxes": boxes, "classes": classes}

In [6]:
# often this cell will produce a red warning that you can ignore
train_ds = tf.data.TFRecordDataset("../shared/seals/train.tfrecord").map(parse_fn).batch(8).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.TFRecordDataset("../shared/seals/val.tfrecord").map(parse_fn).batch(8).prefetch(tf.data.AUTOTUNE)
new_test_ds = tf.data.TFRecordDataset("../shared/seals/test.tfrecord").map(parse_fn).batch(8).prefetch(tf.data.AUTOTUNE)

I0000 00:00:1772602817.857186     480 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Note that the training dataset has had the following augmentation applied before it was saved to the `train.tfrecord` file.

In [7]:
augmenter = keras.Sequential([
    keras_cv.layers.RandomFlip(mode="horizontal", bounding_box_format="xywh"),
    keras_cv.layers.JitteredResize(
        target_size=(640, 640),
        scale_factor=(0.7, 1.3), # Zoom in and out on the seals
        bounding_box_format="xywh"
    ),
])

## Model Set Up

The first thing we'll do is set the anchor boxes. Anchor boxes are fixed sized boxes that the model uses to predict the bounding box for an object [(source)](https://keras.io/examples/vision/retinanet/#:~:text=Implementing%20Anchor%20generator,three%20scales%20and%20three%20ratios). These anchor boxes are specific to the seal dataset and were created using [this code](https://github.com/martinzlocha/anchor-optimization).

In [8]:
# anchors
ratios = [0.488, 1.0, 2.05]
scales = [0.837, 1.032, 1.265]

anchor_generator = keras_cv.layers.AnchorGenerator(
    bounding_box_format="xywh",
    strides=[8, 16, 32, 64, 128], 
    sizes=[32.0, 64.0, 128.0, 256.0, 512.0],
    aspect_ratios=[0.488, 1.0, 2.05], # Changed from 'ratios'
    scales=[0.837, 1.032, 1.265],
)

Next we'll load in the model [backbone](https://huggingface.co/docs/transformers/en/main_classes/backbones#backbone) and define the model architecture.

In [9]:
# pre-trained backbone
backbone = keras_cv.models.ResNet50Backbone.from_preset("resnet50_imagenet")

# retinanet for 3 elephant seal classes
model = keras_cv.models.RetinaNet(
    num_classes=3, 
    backbone=backbone,
    bounding_box_format="xywh", 
    anchor_generator=anchor_generator,
    prediction_decoder=keras_cv.layers.NonMaxSuppression(
        bounding_box_format="xywh",
        from_logits=True, 
        confidence_threshold=0.5, # model has to be at least 50% confident in its detection
        iou_threshold=0.3, # lower value = less overlap between detections
    )
)

In [10]:
print(model.summary())

Model: "retina_net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, None,      │          0 │ -                 │
│                     │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ [(None, None,     │ 23,561,152 │ images[0][0]      │
│ (Functional)        │ None, 512),       │            │                   │
│                     │ (None, None,      │            │                   │
│                     │ None, 1024),      │            │                   │
│                     │ (None, None,      │            │                   │
│                     │ None, 2048)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_pyramid     │ [(None, None,     │  7,997,440 │ functional[0][0], │
│ (FeaturePyramid)    │ None, 256),       │            │ functional[0][1], │
│                     │ (None, None,      │            │ functional[0][2]  │
│                     │ None, 256),       │            │                   │
│                     │ (None, None,      │            │                   │
│                     │ None, 256),       │            │                   │
│                     │ (None, None,      │            │                   │
│                     │ None, 256),       │            │                   │
│                     │ (None, None,      │            │                   │
│                     │ None, 256)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction_head_1   │ (None, None,      │  1,853,220 │ feature_pyramid[… │
│ (PredictionHead)    │ None, 36)         │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction_head     │ (None, None,      │  1,832,475 │ feature_pyramid[… │
│ (PredictionHead)    │ None, 27)         │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
│                     │                   │            │ feature_pyramid[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, None, 4)   │          0 │ prediction_head_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, None, 4)   │          0 │ prediction_head_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_4 (Reshape) │ (None, None, 4)   │          0 │ prediction_head_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_6 (Reshape) │ (None, None, 4)   │          0 │ prediction_head_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_8 (Reshape) │ (None, None, 4)   │          0 │ prediction_head_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, None, 3)   │          0 │ prediction_head[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, None, 3)   │          0 │ prediction_head[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_5 (Reshape) │ (None, None, 3)   │          0 │ prediction_head[… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 35,244,287 (134.45 MB)

 Trainable params: 35,191,167 (134.24 MB)

 Non-trainable params: 53,120 (207.50 KB)

None


## Transfer Learning

Before we get to transfer learning, there's a few set up steps. We first need to define how the model calculates loss, makes updates during training, and where to save the learned weights. 

In [11]:
# provide the model with "rules" for how to calculate loss and make updates during training
model.compile(
    classification_loss="focal",
    box_loss=keras_cv.losses.SmoothL1Loss(), 
    optimizer=keras.optimizers.AdamW(
        learning_rate=0.001,
        weight_decay=0.004,
        global_clipnorm=10.0
    )
)

In [12]:
# folder for checkpoints
checkpoint_dir = "my-model-weights"
os.makedirs(checkpoint_dir, exist_ok=True)

# define the callback
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "epoch-{epoch:02d}.weights.h5"),
    save_weights_only=True,
    monitor="val_loss",
    mode="min",
    save_best_only=False,
    verbose=1
)

Now we take the above the model weights learned from ImageNet as the starting point for the weights of our new model to detect cows, pups, and bulls. The following is only one [epoch](https://www.geeksforgeeks.org/machine-learning/epoch-in-machine-learning/) (a complete pass through the training data) which should take about 5 minutes to run. 

In [ ]:
# train the model on the seals data!
model.fit(
    train_ds, 
    epochs=1, 
    validation_data=val_ds,
    callbacks=[checkpoint_callback]
)

## Evaluation and Inference

You can work with the model weights that you just learned

In [ ]:
checkpoint_path = "my-model-weights/epoch-01.weights.weights.h5"

model.load_weights(checkpoint_path)

Or with model weights we provide after 10 epochs of training by uncommenting the below cell. This model is going to be better than after 1 epoch of training, but is not perfect. 

In [ ]:
# checkpoint_path = "../shared/seals/epoch-10.weights.h5"

# model.load_weights(checkpoint_path)

Let's also bump up the confidence_threshold. This means we don't get low confidence predictions out the model during inference.

In [ ]:
model.prediction_decoder.confidence_threshold = 0.5

In [ ]:
coco_metrics = keras_cv.metrics.BoxCOCOMetrics(
    bounding_box_format="xywh", 
    evaluate_freq=1
)

print("Calculating mAP on test set...")

# Reset the metric state before starting
coco_metrics.reset_state()

for batch in test_ds:
    images, y_true = batch
    
    # Get model predictions
    y_pred = model.predict(images, verbose=0)
    
    # Update the metric with true labels and predicted results
    coco_metrics.update_state(y_true, y_pred)

# 3. Final Result
metrics_result = coco_metrics.result()

print(f"Overall mAP: {metrics_result['MaP']:.4f}")
print(f"mAP at 50% IoU: {metrics_result['MaP@[IoU=50]']:.4f}")
print(f"mAP at 75% IoU: {metrics_result['MaP@[IoU=75]']:.4f}")

mAP @ 50% IoU: is a "lenient" score; if the predicted box overlaps the real box by 50%, it counts as a win. this is a good measure for "does the model know a seal is there?"   
mAP @ 75% IoU: is a "strict" score, more concerned with the precision of the box's placement.  

### Visualization of Inference

We've chosen some images in `shared/seals/test` that are interesting to visualization detections for. You can edit these to be different images if you wish

In [ ]:
image_list = [
    '5MSL3749-174_png.rf.1ade9f0908936fd853dc2ae9a0f00233.jpg',
    '5MSL3764-21_png.rf.d67ea7373c5b15df832b5e11d4d59192.jpg',
    '5MSL3819-65_png.rf.7cf3c5254e14e6b5cf5905c04ef3f848.jpg',
    '5MSL0109-125_png.rf.0999b3e2f7414ce671eb6ece5f074af1.jpg',
    '5MSL0109-4_png.rf.4c177bee4ff3cdfb044448de4af8f62c.jpg',
    '5MSL0087-138_png.rf.6843d941557d06ab666d79fe30071175.jpg',
    '5MSL4416-99_png.rf.1e915260973f7fd233ef130506a02836.jpg',
    '4MSL0120-163_png.rf.fec5aded76dfdbf287db0a33cffecb65.jpg',]

In [ ]:
# create the full path to these images
full_image_list_paths = ["shared/seals/test/" + img for img in image_list]

In [ ]:
# prepare images
def load_and_prep(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(640, 640))
    return tf.keras.utils.img_to_array(img)

images = np.array([load_and_prep(img) for img in full_image_list_paths])

# run inference
# model returns a dict with 'boxes' and 'classes'
predictions = model.predict(images)

# map between model labels and human-readable labels
class_mapping = {0: "Bull", 1: "Cow", 2: "Pup"}

# plot the results
keras_cv.visualization.plot_bounding_box_gallery(
    images,
    value_range=(0, 255),
    rows=4,
    cols=2, 
    y_pred=filtered_predictions,
    bounding_box_format="xywh", 
    scale=5,
    font_scale=0.7,
    line_thickness=2,
    class_mapping=class_mapping,
)

plt.show()